# Del 1 – Grundläggande analys och statistik

Denna notebook innehåller:
- Beskrivande statistik för `age`, `weight`, `height`, `systolic_bp`, `cholesterol`
- Tre grafer (histogram, boxplot per kön, stapeldiagram för andel rökare)
- Simulering av sjukdomsandel (1000 personer) jämfört med verklig andel i datasetet
- Konfidensintervall för medelvärdet av `systolic_bp` (normalapproximation och bootstrap)
- Hypotesprövning: *"Rökare har högre medel-blodtryck än icke-rökare."* (one-sided permutation test)
- (VG) Jämförelse av CI-metoder och power-approximation

**Dataset:** `health_study_dataset.csv`  
**Kolumner i datasetet:** `id, age, sex, height, weight, systolic_bp, cholesterol, smoker, disease`

## Metodval (kort motivering)

- **Beskrivande statistik:** Medel, median, min och max ger en snabb översikt över läge och spridning samt outliers.
- **Grafer:** Histogram (för fördelning), boxplot per kön (jämförelse mellan grupper) och stapeldiagram (andelar) ger kompletterande visuella insikter.
- **Simulering:** Vi uppskattar den verkliga sjukdomssannolikheten `p` från datasetet och simulerar 1000 Bernoulli-drag med samma `p` för att jämföra andelar.
- **Konfidensintervall:** Vi använder både normalapproximation (snabb, fungerar väl för stora n) och bootstrap (icke-parametrisk) för att få en robust jämförelse.
- **Hypotesprövning (rökare vs icke-rökare):** En enkel och robust *permutations-/bootstrap-baserad* test av skillnad i medelvärde. Vi testar ensidigt om rökare har högre medel-BP.
- **Power (VG):** Vi uppskattar power med en z-approximation baserat på observerad effektstorlek och gruppstorlekar/varians från data.

> Referenser: Se kursmaterialet samt dokumentationen för NumPy, pandas och matplotlib för funktioner och syntax.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Läs in data
df = pd.read_csv("/mnt/data/health_study_dataset.csv")
df.head()

In [ ]:
# Beskrivande statistik
expected_numeric = ["age", "weight", "height", "systolic_bp", "cholesterol"]
present_numeric = [c for c in expected_numeric if c in df.columns]

desc_stats = {}
for c in present_numeric:
    s = df[c].dropna().values
    if len(s) > 0:
        desc_stats[c] = {
            "n": len(s),
            "mean": float(np.mean(s)),
            "median": float(np.median(s)),
            "min": float(np.min(s)),
            "max": float(np.max(s))
        }
pd.DataFrame(desc_stats).T

In [ ]:
# Plots
candidate_smoker_cols = [c for c in df.columns if c.lower() in ("smoker", "is_smoker", "smokes", "smoking", "smoker_flag")]
smoker_col = candidate_smoker_cols[0] if candidate_smoker_cols else None
candidate_sex_cols = [c for c in df.columns if c.lower() in ("sex", "gender")]
sex_col = candidate_sex_cols[0] if candidate_sex_cols else None

# 1) Histogram över systolic_bp
if "systolic_bp" in df.columns:
    plt.figure()
    df["systolic_bp"].dropna().plot(kind="hist", bins=20, edgecolor="black")
    plt.title("Histogram of Systolic Blood Pressure")
    plt.xlabel("Systolic BP")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# 2) Boxplot över vikt per kön
if ("weight" in df.columns) and (sex_col is not None):
    plt.figure()
    order = sorted(df[sex_col].dropna().astype(str).unique())
    data = [df.loc[df[sex_col].astype(str) == k, "weight"].dropna() for k in order]
    plt.boxplot(data, labels=order, showmeans=True)
    plt.title(f"Weight by {sex_col}")
    plt.xlabel(sex_col)
    plt.ylabel("Weight")
    plt.tight_layout()
    plt.show()

# 3) Stapeldiagram över andelen rökare
def to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    true_vals = {"1", "true", "yes", "y", "t", "smoker", "ja"}
    return s.isin(true_vals).astype(int)

if smoker_col is not None:
    smoker_bin = to_binary(df[smoker_col])
    plt.figure()
    counts = smoker_bin.value_counts().sort_index()
    props = counts / counts.sum()
    props.index = props.index.map({0: "Non-smoker", 1: "Smoker"})
    props.plot(kind="bar", edgecolor="black")
    plt.title("Proportion of Smokers")
    plt.xlabel("Smoking status")
    plt.ylabel("Proportion")
    plt.tight_layout()
    plt.show()

In [ ]:
# Simulering: sjukdomsandel
candidate_disease_cols = [c for c in df.columns if c.lower() in ("disease", "has_disease", "condition", "ill", "diagnosis")]
disease_col = candidate_disease_cols[0] if candidate_disease_cols else None

def to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    true_vals = {"1", "true", "yes", "y", "t", "smoker", "ja"}
    return s.isin(true_vals).astype(int)

if disease_col is not None:
    p_real = to_binary(df[disease_col]).dropna().mean()
    np.random.seed(42)
    sim = np.random.binomial(1, p_real, size=1000)
    p_sim = sim.mean()
    print({"real_proportion_disease": p_real, "simulated_proportion_disease": p_sim, "n_sim": 1000})
else:
    print("Ingen sjukdomskolumn hittades.")

In [ ]:
# Konfidensintervall för mean(systolic_bp)
import numpy as np

def normal_approx_ci_mean(x, alpha=0.05):
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1)
    z = 1.959963984540054
    half = z * std / np.sqrt(n)
    return mean - half, mean + half

def bootstrap_ci_mean(x, B=5000, alpha=0.05, seed=123):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    n = len(x)
    boot = np.empty(B)
    for b in range(B):
        samp = rng.choice(x, size=n, replace=True)
        boot[b] = samp.mean()
    return np.quantile(boot, alpha/2), np.quantile(boot, 1 - alpha/2)

if "systolic_bp" in df.columns:
    bp = df["systolic_bp"].dropna().values
    ci_norm = normal_approx_ci_mean(bp, alpha=0.05)
    ci_boot = bootstrap_ci_mean(bp, B=5000, alpha=0.05, seed=123)
    print({"Normal approx (95%)": ci_norm, "Bootstrap (95%)": ci_boot})
else:
    print("Kolumnen 'systolic_bp' saknas.")

In [ ]:
# Hypotesprövning: Rökare har högre mean BP än icke-rökare (ensidigt)
def to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    true_vals = {"1", "true", "yes", "y", "t", "smoker", "ja"}
    return s.isin(true_vals).astype(int)

candidate_smoker_cols = [c for c in df.columns if c.lower() in ("smoker", "is_smoker", "smokes", "smoking", "smoker_flag")]
smoker_col = candidate_smoker_cols[0] if candidate_smoker_cols else None

def bootstrap_one_sided_test_greater(x, y, B=5000, seed=99):
    import numpy as np
    rng = np.random.default_rng(seed)
    x = np.asarray(x); y = np.asarray(y)
    x = x[~np.isnan(x)]; y = y[~np.isnan(y)]
    obs = x.mean() - y.mean()
    pooled = np.concatenate([x, y])
    n_x, n_y = len(x), len(y)
    cnt = 0
    for b in range(B):
        perm = rng.permutation(pooled)
        x_star, y_star = perm[:n_x], perm[n_x:]
        diff = x_star.mean() - y_star.mean()
        if diff >= obs:
            cnt += 1
    pval = (cnt + 1) / (B + 1)
    return obs, pval

if (smoker_col is not None) and ("systolic_bp" in df.columns):
    smk = df.loc[to_binary(df[smoker_col]) == 1, "systolic_bp"].astype(float).values
    non = df.loc[to_binary(df[smoker_col]) == 0, "systolic_bp"].astype(float).values
    if len(smk) > 5 and len(non) > 5:
        obs_diff, pval = bootstrap_one_sided_test_greater(smk, non, B=5000, seed=99)
        print({"observed_mean_diff_smoker_minus_non": obs_diff, "p_value_one_sided": pval})
    else:
        print("För få observationer i någon grupp för att testa.")
else:
    print("Saknar 'smoker'-kolumn eller 'systolic_bp'.")

In [ ]:
# Power-approximation (VG): z-approx för ensidigt test
import math

def normal_cdf(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

candidate_smoker_cols = [c for c in df.columns if c.lower() in ("smoker", "is_smoker", "smokes", "smoking", "smoker_flag")]
smoker_col = candidate_smoker_cols[0] if candidate_smoker_cols else None

if (smoker_col is not None) and ("systolic_bp" in df.columns):
    sm = df.loc[to_binary(df[smoker_col]) == 1, "systolic_bp"].dropna().astype(float).values
    nn = df.loc[to_binary(df[smoker_col]) == 0, "systolic_bp"].dropna().astype(float).values
    if len(sm) > 10 and len(nn) > 10:
        n1, n0 = len(sm), len(nn)
        s1, s0 = sm.std(ddof=1), nn.std(ddof=1)
        d_obs = sm.mean() - nn.mean()
        se = math.sqrt((s1**2)/n1 + (s0**2)/n0)
        alpha = 0.05
        z_crit = 1.6448536269514722
        mu_z = d_obs / se
        power = 1 - normal_cdf(z_crit - mu_z)
        print({"n_smokers": n1, "n_non_smokers": n0, "std_smokers": s1, "std_non_smokers": s0,
                "observed_mean_diff": d_obs, "alpha": alpha, "approx_one_sided_power": power})
    else:
        print("För få observationer för en rimlig power-approximation.")
else:
    print("Saknar 'smoker'-kolumn eller 'systolic_bp'.")

## Kort tolkning av resultaten (exempel)

- **Konfidensintervall:** Bootstrap och normalapproximation bör ligga relativt nära varandra om fördelningen inte är alltför sned. Skillnader kan indikera outliers eller icke-normalitet.
- **Hypotesprövning:** Ett lågt p-värde (< 0.05, ensidigt) talar för att rökare har högre medel-BP än icke-rökare i detta data. Annars saknas evidens för en sådan skillnad.
- **Power:** Högre power betyder högre chans att upptäcka en verklig skillnad. Den beror på effektstorlek, varians och stickprovsstorlek.